# Raw Data Formatting

This notebook builds the base annual dataset used throughout the project. The purpose is to pull Indonesia-specific observations from each raw source file, align every indicator to the same year index, and create one merged table that can be cleaned and modelled later.


In [32]:
import pandas as pd
import pandas as pd

gdp = pd.read_csv(
    "gdp.csv",  
    skiprows=4,             # Skip source metadata rows so the first real data row becomes the header.
    encoding="utf-8-sig"    # Preserve the file encoding used by the source export.
)

# Drop the empty column introduced by the source export format.
gdp = gdp.loc[:, ~gdp.columns.str.contains(r"^Unnamed")]


In [34]:
# Keep only Indonesia's GDP growth history and reshape it into a year-by-value table.
indonesia_gdp_growth = gdp[gdp["Country Name"] == "Indonesia"].iloc[:, 24:69]
indonesia_gdp_growth = indonesia_gdp_growth.transpose()
indonesia_gdp_growth.columns = ["gdp_growth(%)"]
indonesia_gdp_growth.index.name = "year"
indonesia_gdp_growth.reset_index(inplace=True)
indonesia_gdp_growth.head()


,year,gdp_growth(%)
0,1980,9.880078
1,1981,7.927157
2,1982,2.246445
3,1983,4.192967
4,1984,6.975528


In [75]:
# Create lagged GDP growth columns so later notebooks can model current and next-year growth.
indonesia_gdp_growth["gdp_growth_lag1(%)"] = indonesia_gdp_growth["gdp_growth(%)"].shift(1)
indonesia_gdp_growth["gdp_growth_lag2(%)"] = indonesia_gdp_growth["gdp_growth(%)"].shift(2)
indonesia_gdp_growth.head()


,year,gdp_growth(%),gdp_growth_lag1(%),gdp_growth_lag2(%)
0,1980,9.880078,NaN,NaN
1,1981,7.927157,9.880078,NaN
2,1982,2.246445,7.927157,9.880078
3,1983,4.192967,2.246445,7.927157
4,1984,6.975528,4.192967,2.246445


## Household Consumption Growth

This section adds household consumption growth because private consumption is a major driver of Indonesia's GDP. The goal is to merge it into the main year-level table so it can be compared directly with GDP growth.


This merge keeps the raw pipeline simple: read the indicator, isolate Indonesia, reshape it into a year-indexed series, and join it to the GDP base table.


In [43]:
household_expenditure = pd.read_csv(
    "consumption_expenditure.csv",
    skiprows=4,
    encoding="utf-8-sig"
)

# Drop the empty column introduced by the source export format.
household_expenditure = household_expenditure.loc[:, ~household_expenditure.columns.str.contains(r"^Unnamed")]
#select Indonesia's household consumption data from 1980 to 2025
household_expenditure= household_expenditure[household_expenditure["Country Name"] == "Indonesia"].iloc[:, 24:69]
household_expenditure = household_expenditure.transpose()
household_expenditure.columns = ["household_expenditure(%)"]
household_expenditure.head


<bound method NDFrame.head of       household_expenditure(%)
1980                 70.834204
1981                 76.531919
1982                 81.334706
1983                 71.666649
1984                 72.352702
1985                 72.424958
1986                 75.208760
1987                 71.411564
1988                 72.114454
1989                 69.343427
1990                 73.024205
1991                 73.359707
1992                 72.356220
1993                 67.535307
1994                 67.796950
1995                 69.406120
1996                 69.924141
1997                 68.524069
1998                 73.474960
1999                 80.548254
2000                 68.182366
2001                 70.039221
2002                 74.879695
2003                 76.267506
2004                 75.090350
2005                 72.471998
2006                 71.296226
2007                 71.889159
2008                 69.045034
2009                 68.291934
2010     

In [44]:
# Join household consumption to the GDP base table using the shared year index.
merged_data = pd.merge(indonesia_gdp_growth, household_expenditure, left_on="year", right_index=True)
merged_data


,year,gdp_growth(%),gdp_growth_lag1(%),gdp_growth_lag2(%),household_expenditure(%)
0,1980,9.880078,NaN,NaN,70.834204
1,1981,7.927157,9.880078,NaN,76.531919
2,1982,2.246445,7.927157,9.880078,81.334706
3,1983,4.192967,2.246445,7.927157,71.666649
4,1984,6.975528,4.192967,2.246445,72.352702
5,1985,2.462144,6.975528,4.192967,72.424958
6,1986,5.875045,2.462144,6.975528,75.208760
7,1987,4.925927,5.875045,2.462144,71.411564
8,1988,5.780498,4.925927,5.875045,72.114454
9,1989,7.456587,5.780498,4.925927,69.343427


## Unemployment Rate

This section adds labour market information. Unemployment is included because it helps explain whether GDP growth is being supported by a stronger domestic economy or weakened by slack in the job market.


In [58]:
uneploymemnt_rate = pd.read_csv(
    "unemployment_rate.csv",
    skiprows=4,
    encoding="utf-8-sig"
)
# Drop the empty column introduced by the source export format.
uneploymemnt_rate = uneploymemnt_rate.loc[:, ~uneploymemnt_rate.columns.str.contains(r"^Unnamed")]
#select Indonesia's unemployment rate data from 1980 to 2024
uneploymemnt_rate = uneploymemnt_rate[uneploymemnt_rate["Country Name"] == "Indonesia"].iloc[:, 24:68]
uneploymemnt_rate = uneploymemnt_rate.transpose()
uneploymemnt_rate.columns = ["unemployment_rate(%)"]
uneploymemnt_rate.head()


,unemployment_rate(%)
1980,NaN
1981,NaN
1982,NaN
1983,NaN
1984,NaN


In [ ]:
# Add unemployment data to the growing year-level dataset.
merged_data = pd.merge(merged_data, uneploymemnt_rate, left_on="year", right_index=True)
merged_data


## Under-Earning Percentage

This section adds a measure of labour underutilization. It complements the unemployment rate by capturing workers who are employed but still earning below the threshold, which can signal household-side economic stress.


In [79]:
under_earning_percentaege = pd.read_csv(
    "earning_less_than_3$_percentage.csv",
    skiprows=4,
    encoding="utf-8-sig"
)
# Drop the empty column introduced by the source export format.
under_earning_percentaege = under_earning_percentaege.loc[:, ~under_earning_percentaege.columns.str.contains(r"^Unnamed")]
#select Indonesia's under earning percentage data from 1980 to 2024
under_earning_percentaege = under_earning_percentaege[under_earning_percentaege["Country Name"] == "Indonesia"].iloc[:, 24:68]
under_earning_percentaege = under_earning_percentaege.transpose()
under_earning_percentaege.columns = ["under_earning_percentage(%)"]


In [83]:
# Join the under-earning series to the existing merged dataset.
merged_data = pd.merge(indonesia_gdp_growth, household_consumption, left_on="year", right_index=True)
merged_data = pd.merge(merged_data, uneploymemnt_rate, left_on="year", right_index=True)
merged_data = pd.merge(merged_data, under_earning_percentaege, left_on="year", right_index=True)
merged_data.head()


,year,gdp_growth(%),gdp_growth_lag1(%),gdp_growth_lag2(%),household_consumption(%),unemployment_rate(%),under_earning_percentage(%)
0,1980,9.880078,NaN,NaN,70.834204,NaN,NaN
1,1981,7.927157,9.880078,NaN,76.531919,NaN,NaN
2,1982,2.246445,7.927157,9.880078,81.334706,NaN,NaN
3,1983,4.192967,2.246445,7.927157,71.666649,NaN,NaN
4,1984,6.975528,4.192967,2.246445,72.352702,NaN,86.2


## Inflation Rate

Inflation is merged next because price pressure affects real purchasing power, interest-rate conditions, and business sentiment. Including it here makes it available for later feature engineering and modelling.


In [84]:
inflation_rate = pd.read_csv(
    "inflation_rate.csv",
    skiprows=4,
    encoding="utf-8-sig"
)
# Drop the empty column introduced by the source export format.
inflation_rate = inflation_rate.loc[:, ~inflation_rate.columns.str.contains(r"^Unnamed")]
#select Indonesia's inflation rate data from 1980 to 2024
inflation_rate = inflation_rate[inflation_rate["Country Name"] == "Indonesia"].iloc[:, 24:68]
inflation_rate = inflation_rate.transpose()
inflation_rate.columns = ["inflation_rate(%)"]


In [86]:
# Add inflation to the merged macroeconomic panel.
merged_data = pd.merge(merged_data, inflation_rate, left_on="year", right_index=True)
merged_data.head()


,year,gdp_growth(%),gdp_growth_lag1(%),gdp_growth_lag2(%),household_consumption(%),unemployment_rate(%),under_earning_percentage(%),inflation_rate(%)
0,1980,9.880078,NaN,NaN,70.834204,NaN,NaN,18.035422
1,1981,7.927157,9.880078,NaN,76.531919,NaN,NaN,12.265908
2,1982,2.246445,7.927157,9.880078,81.334706,NaN,NaN,9.445425
3,1983,4.192967,2.246445,7.927157,71.666649,NaN,NaN,11.799738
4,1984,6.975528,4.192967,2.246445,72.352702,NaN,86.2,10.455039


## FDI Net Flows

Foreign direct investment is added as a proxy for external capital entering the economy. This helps test whether investment inflows move with later GDP growth.


In [102]:
fdi_net_flows = pd.read_csv(
    "fdi_net_flows.csv",
    skiprows=4,
    encoding="utf-8-sig"
)
# Drop the empty column introduced by the source export format.
fdi_net_flows = fdi_net_flows.loc[:, ~fdi_net_flows.columns.str.contains(r"^Unnamed")]
#select Indonesia's fdi net flows data from 1980 to 2024
fdi_net_flows = fdi_net_flows[fdi_net_flows["Country Name"] == "Indonesia"].iloc[:, 24:68]
fdi_net_flows = fdi_net_flows.transpose()
fdi_net_flows.columns = ["fdi_net_flows(US$)"]
fdi_net_flows.head()


,fdi_net_flows(US$)
1980,180000000.0
1981,133000000.0
1982,225000000.0
1983,292000000.0
1984,222000000.0


In [103]:
merged_data = pd.merge(merged_data, fdi_net_flows, left_on="year", right_index=True)

## Lending Interest Rate

This section adds the policy and financing side of the macro picture. Lending rates are useful because borrowing costs can influence consumption, business expansion, and investment.


In [98]:
lending_interest_rate = pd.read_csv(
    "lending_interest_rate.csv",
    skiprows=4,
    encoding="utf-8-sig"
)
# Drop the empty column introduced by the source export format.
lending_interest_rate = lending_interest_rate.loc[:, ~lending_interest_rate.columns.str.contains(r"^Unnamed")]
#select Indonesia's lending interest rate data from 1980 to 2024
lending_interest_rate = lending_interest_rate[lending_interest_rate["Country Name"] == "Indonesia"].iloc[:, 24:68]
lending_interest_rate = lending_interest_rate.transpose()
lending_interest_rate.columns = ["lending_interest_rate(%)"]
lending_interest_rate.head()


,lending_interest_rate(%)
1980,NaN
1981,NaN
1982,NaN
1983,NaN
1984,NaN


In [ ]:
merged_data = pd.merge(merged_data, lending_interest_rate, left_on="year", right_index=True)
merged_data[10:20]

## Tax Revenue

Tax revenue is merged as a fiscal signal. It can reflect the government's revenue base and broader economic activity, although it also introduces missing-value problems that are handled later.


In [107]:
tax_revenue_percentage = pd.read_csv(
    "tax_revenue(%_total_gdp).csv",
    skiprows=4,
    encoding="utf-8-sig"
)
# Drop the empty column introduced by the source export format.
tax_revenue_percentage = tax_revenue_percentage.loc[:, ~tax_revenue_percentage.columns.str.contains     (r"^Unnamed")]
#select Indonesia's tax revenue percentage data from 1980 to 2024
tax_revenue_percentage = tax_revenue_percentage[tax_revenue_percentage["Country Name"] == "Indonesia"].iloc[:, 24:68]
tax_revenue_percentage = tax_revenue_percentage.transpose()
tax_revenue_percentage.columns = ["tax_revenue_percentage(%)"]  


In [109]:
merged_data = pd.merge(merged_data, tax_revenue_percentage, left_on="year", right_index=True)
merged_data[10:20]

,year,gdp_growth(%),gdp_growth_lag1(%),gdp_growth_lag2(%),household_consumption(%),unemployment_rate(%),under_earning_percentage(%),inflation_rate(%),lending_interest_rate(%),fdi_net_flows(US$),tax_revenue_percentage(%)
10,1990,7.242132,7.456587,5.780498,73.024205,NaN,78.6,7.819198,20.825000,1.093000e+09,19.137084
11,1991,6.911983,7.242132,7.456587,73.359707,2.617,NaN,9.419068,25.533333,1.482000e+09,17.189697
12,1992,6.497507,6.911983,7.242132,72.356220,2.734,NaN,7.523510,24.033333,1.777000e+09,17.122953
13,1993,6.496408,6.497507,6.911983,67.535307,2.782,78.0,9.671911,20.586667,2.004000e+09,14.356420
14,1994,7.539971,6.496408,6.497507,67.796950,4.366,NaN,8.531998,17.760000,2.109000e+09,15.948419
15,1995,8.220007,7.539971,6.496408,69.406120,4.611,NaN,9.420303,18.851667,4.346000e+09,14.964772
16,1996,7.818187,8.220007,7.539971,69.924141,4.861,68.1,7.973281,19.217500,6.194000e+09,14.234802
17,1997,4.699879,7.818187,8.220007,68.524069,4.684,NaN,6.226152,21.817500,4.677000e+09,16.011848
18,1998,-13.126725,4.699879,7.818187,73.474960,5.459,82.8,58.451041,32.154167,-2.408000e+08,15.027588
19,1999,0.791126,-13.126725,4.699879,80.548254,6.358,66.2,20.477832,27.662500,-1.865621e+09,16.315836


## State Revenue

This section adds state revenue to extend the fiscal view beyond tax revenue alone. The objective is to capture whether public-sector revenue conditions move with growth outcomes.


State revenue is kept in the same year-level format so it can be combined cleanly with the other macro indicators.


In [111]:
state_revenues_percentage = pd.read_csv(
    "state_revenue.csv",
    skiprows=4,
    encoding="utf-8-sig"
)
# Drop the empty column introduced by the source export format.
state_revenues_percentage = state_revenues_percentage.loc[:, ~state_revenues_percentage.columns.str.contains(r"^Unnamed")]
#select Indonesia's state revenues percentage data from 1980 to 2024
state_revenues_percentage = state_revenues_percentage[state_revenues_percentage["Country Name"] == "Indonesia"].iloc[:, 24:68]
state_revenues_percentage = state_revenues_percentage.transpose()
state_revenues_percentage.columns = ["state_revenues_percentage(%)"]
state_revenues_percentage.head()


,state_revenues_percentage(%)
1980,22.897656
1981,25.474300
1982,21.489923
1983,21.046818
1984,21.508291


In [114]:
merged_data = pd.merge(merged_data, state_revenues_percentage, left_on="year", right_index=True)
merged_data[10:20]

,year,gdp_growth(%),gdp_growth_lag1(%),gdp_growth_lag2(%),household_consumption(%),unemployment_rate(%),under_earning_percentage(%),inflation_rate(%),lending_interest_rate(%)_x,fdi_net_flows(US$),tax_revenue_percentage(%),state_revenues_percentage(%)
10,1990,7.242132,7.456587,5.780498,73.024205,NaN,78.6,7.819198,20.825000,1.093000e+09,19.137084,20.228306
11,1991,6.911983,7.242132,7.456587,73.359707,2.617,NaN,9.419068,25.533333,1.482000e+09,17.189697,18.648038
12,1992,6.497507,6.911983,7.242132,72.356220,2.734,NaN,7.523510,24.033333,1.777000e+09,17.122953,19.487503
13,1993,6.496408,6.497507,6.911983,67.535307,2.782,78.0,9.671911,20.586667,2.004000e+09,14.356420,17.077663
14,1994,7.539971,6.496408,6.497507,67.796950,4.366,NaN,8.531998,17.760000,2.109000e+09,15.948419,18.155527
15,1995,8.220007,7.539971,6.496408,69.406120,4.611,NaN,9.420303,18.851667,4.346000e+09,14.964772,17.691860
16,1996,7.818187,8.220007,7.539971,69.924141,4.861,68.1,7.973281,19.217500,6.194000e+09,14.234802,16.953516
17,1997,4.699879,7.818187,8.220007,68.524069,4.684,NaN,6.226152,21.817500,4.677000e+09,16.011848,18.137719
18,1998,-13.126725,4.699879,7.818187,73.474960,5.459,82.8,58.451041,32.154167,-2.408000e+08,15.027588,16.466714
19,1999,0.791126,-13.126725,4.699879,80.548254,6.358,66.2,20.477832,27.662500,-1.865621e+09,16.315836,18.059979


## Gross Capital Formation Growth

Gross capital formation is included as a proxy for capital investment and productive capacity expansion. It is one of the clearest indicators for testing whether investment momentum supports future GDP growth.


In [115]:
gcf_growth = pd.read_csv(
    "gross_capital_formation_growth_annual.csv",
    skiprows=4,
    encoding="utf-8-sig"
)
# Drop the empty column introduced by the source export format.
gcf_growth = gcf_growth.loc[:, ~gcf_growth.columns.str.contains(r"^Unnamed")]
#select Indonesia's gross capital formation growth data from 1980 to 2024
gcf_growth = gcf_growth[gcf_growth["Country Name"] == "Indonesia"].iloc[:, 24:68]
gcf_growth = gcf_growth.transpose()
gcf_growth.columns = ["gross_capital_formation_growth(%)"]
gcf_growth.head()


,gross_capital_formation_growth(%)
1980,24.369558
1981,12.398103
1982,5.647824
1983,627.499564
1984,52.568955


In [146]:
merged_data = pd.merge(merged_data, gcf_growth, left_on="year", right_index=True)
merged_data[10:30]

,year,gdp_growth(%),gdp_growth_lag1(%),gdp_growth_lag2(%),household_consumption(%),unemployment_rate(%),under_earning_percentage(%),inflation_rate(%),lending_interest_rate(%)_x,fdi_net_flows(US$),tax_revenue_percentage(%),state_revenues_percentage(%),gross_capital_formation_growth(%)_x,import_of_goods_and_services(US$),export_of_goods_and_services(US$),gross_capital_formation_growth(%)_y
10,1990,7.242132,7.456587,5.780498,73.024205,NaN,78.6,7.819198,20.825000,1.093000e+09,19.137084,20.228306,103.268736,2.715728e+10,2.898253e+10,103.268736
11,1991,6.911983,7.242132,7.456587,73.359707,2.617,NaN,9.419068,25.533333,1.482000e+09,17.189697,18.648038,-33.202098,3.089119e+10,3.306381e+10,-33.202098
12,1992,6.497507,6.911983,7.242132,72.356220,2.734,NaN,7.523510,24.033333,1.777000e+09,17.122953,19.487503,13.744395,3.472088e+10,3.880173e+10,13.744395
13,1993,6.496408,6.497507,6.911983,67.535307,2.782,78.0,9.671911,20.586667,2.004000e+09,14.356420,17.077663,38.450762,3.755594e+10,4.227440e+10,38.450762
14,1994,7.539971,6.496408,6.497507,67.796950,4.366,NaN,8.531998,17.760000,2.109000e+09,15.948419,18.155527,36.400063,4.486988e+10,4.689663e+10,36.400063
15,1995,8.220007,7.539971,6.496408,69.406120,4.611,NaN,9.420303,18.851667,4.346000e+09,14.964772,17.691860,7.800788,5.588228e+10,5.318531e+10,7.800788
16,1996,7.818187,8.220007,7.539971,69.924141,4.861,68.1,7.973281,19.217500,6.194000e+09,14.234802,16.953516,-52.080132,6.011698e+10,5.871720e+10,-52.080132
17,1997,4.699879,7.818187,8.220007,68.524069,4.684,NaN,6.226152,21.817500,4.677000e+09,16.011848,18.137719,-25.772651,6.070015e+10,6.010604e+10,-25.772651
18,1998,-13.126725,4.699879,7.818187,73.474960,5.459,82.8,58.451041,32.154167,-2.408000e+08,15.027588,16.466714,-164.509353,4.124971e+10,5.055573e+10,-164.509353
19,1999,0.791126,-13.126725,4.699879,80.548254,6.358,66.2,20.477832,27.662500,-1.865621e+09,16.315836,18.059979,85.726670,3.840207e+10,4.972026e+10,85.726670


## Imports

Import values are added before exports so the project can later derive an export-import ratio. This gives a compact way to describe external trade balance conditions.


In [137]:
import_of_goods_and_services = pd.read_csv(
    "import_goods_and_services.csv",
    skiprows=4,
    encoding="utf-8-sig"
)
# Drop the empty column introduced by the source export format.
import_of_goods_and_services = import_of_goods_and_services.loc[:, ~import_of_goods_and_services.columns.str.contains(r"^Unnamed")]
#select Indonesia's import of goods and services data from 1980 to 2024
import_of_goods_and_services = import_of_goods_and_services[import_of_goods_and_services["Country Name"] == "Indonesia"].iloc[:, 24:68]
import_of_goods_and_services = import_of_goods_and_services.transpose()
import_of_goods_and_services.columns = ["import_of_goods_and_services(US$)"]
import_of_goods_and_services.head()


,import_of_goods_and_services(US$)
1980,1.607649e+10
1981,2.184722e+10
1982,2.370914e+10
1983,2.335427e+10
1984,1.934295e+10


The import and export sections are intentionally paired because they are used together again during feature engineering.


## Export Value

This section completes the trade block of the dataset. Export values are important both on their own and as part of the later export-import ratio feature.


In [140]:
export_of_goods_and_services = pd.read_csv(
    "export_of_good_and_service.csv",
    skiprows=4,
    encoding="utf-8-sig"
)
# Drop the empty column introduced by the source export format.
export_of_goods_and_services = export_of_goods_and_services.loc[:, ~export_of_goods_and_services.columns.str.contains(r"^Unnamed")]
#select Indonesia's export of goods and services data from 1980 to 2024
export_of_goods_and_services = export_of_goods_and_services[export_of_goods_and_services["Country Name"] == "Indonesia"].iloc[:, 24:68]
export_of_goods_and_services = export_of_goods_and_services.transpose()
export_of_goods_and_services.columns = ["export_of_goods_and_services(US$)"]
export_of_goods_and_services.head()


,export_of_goods_and_services(US$)
1980,2.208839e+10
1981,2.362907e+10
1982,2.017659e+10
1983,2.248829e+10
1984,2.317777e+10


In [141]:
merged_data = pd.merge(merged_data, import_of_goods_and_services, left_on="year", right_index=True)
merged_data = pd.merge(merged_data, export_of_goods_and_services, left_on="year", right_index=True)

## Exchange Rate

The final raw indicator is the rupiah exchange rate against the US dollar. This is included because currency pressure can influence inflation, trade competitiveness, and investor sentiment.


In [147]:
exchange_rate = pd.read_csv(
    "exchange_rate.csv",
    skiprows=4,
    encoding="utf-8-sig"
)
# Drop the empty column introduced by the source export format.
exchange_rate = exchange_rate.loc[:, ~exchange_rate.columns.str.contains(r"^Unnamed")]
#select Indonesia's exchange rate data from 1980 to 2024
exchange_rate = exchange_rate[exchange_rate["Country Name"] == "Indonesia"].iloc[:, 24:68]
exchange_rate = exchange_rate.transpose()
exchange_rate.columns = ["exchange_rate(IDR/US$)"]
exchange_rate.head()


,exchange_rate(IDR/US$)
1980,626.994000
1981,631.756667
1982,661.420750
1983,909.264833
1984,1025.944833


In [148]:
merged_data = pd.merge(merged_data,  exchange_rate, left_on="year", right_index=True)
merged_data[10:20]

,year,gdp_growth(%),gdp_growth_lag1(%),gdp_growth_lag2(%),household_consumption(%),unemployment_rate(%),under_earning_percentage(%),inflation_rate(%),lending_interest_rate(%)_x,fdi_net_flows(US$),tax_revenue_percentage(%),state_revenues_percentage(%),gross_capital_formation_growth(%)_x,import_of_goods_and_services(US$),export_of_goods_and_services(US$),gross_capital_formation_growth(%)_y,exchange_rate(IDR/US$)
10,1990,7.242132,7.456587,5.780498,73.024205,NaN,78.6,7.819198,20.825000,1.093000e+09,19.137084,20.228306,103.268736,2.715728e+10,2.898253e+10,103.268736,1842.813333
11,1991,6.911983,7.242132,7.456587,73.359707,2.617,NaN,9.419068,25.533333,1.482000e+09,17.189697,18.648038,-33.202098,3.089119e+10,3.306381e+10,-33.202098,1950.317500
12,1992,6.497507,6.911983,7.242132,72.356220,2.734,NaN,7.523510,24.033333,1.777000e+09,17.122953,19.487503,13.744395,3.472088e+10,3.880173e+10,13.744395,2029.920833
13,1993,6.496408,6.497507,6.911983,67.535307,2.782,78.0,9.671911,20.586667,2.004000e+09,14.356420,17.077663,38.450762,3.755594e+10,4.227440e+10,38.450762,2087.103867
14,1994,7.539971,6.496408,6.497507,67.796950,4.366,NaN,8.531998,17.760000,2.109000e+09,15.948419,18.155527,36.400063,4.486988e+10,4.689663e+10,36.400063,2160.753675
15,1995,8.220007,7.539971,6.496408,69.406120,4.611,NaN,9.420303,18.851667,4.346000e+09,14.964772,17.691860,7.800788,5.588228e+10,5.318531e+10,7.800788,2248.607975
16,1996,7.818187,8.220007,7.539971,69.924141,4.861,68.1,7.973281,19.217500,6.194000e+09,14.234802,16.953516,-52.080132,6.011698e+10,5.871720e+10,-52.080132,2342.296292
17,1997,4.699879,7.818187,8.220007,68.524069,4.684,NaN,6.226152,21.817500,4.677000e+09,16.011848,18.137719,-25.772651,6.070015e+10,6.010604e+10,-25.772651,2909.380000
18,1998,-13.126725,4.699879,7.818187,73.474960,5.459,82.8,58.451041,32.154167,-2.408000e+08,15.027588,16.466714,-164.509353,4.124971e+10,5.055573e+10,-164.509353,10013.622500
19,1999,0.791126,-13.126725,4.699879,80.548254,6.358,66.2,20.477832,27.662500,-1.865621e+09,16.315836,18.059979,85.726670,3.840207e+10,4.972026e+10,85.726670,7855.150000


In [150]:
# Convert the year field back to an integer so it stays consistent across later notebooks.
merged_data["year"] = pd.to_datetime(merged_data["year"], format="%Y").dt.year
merged_data.info()


<class 'pandas.core.frame.DataFrame'>
Index: 44 entries, 0 to 43
Data columns (total 17 columns):
 #   Column                               Non-Null Count  Dtype  
---  ------                               --------------  -----  
 0   year                                 44 non-null     int32  
 1   gdp_growth(%)                        44 non-null     float64
 2   gdp_growth_lag1(%)                   43 non-null     float64
 3   gdp_growth_lag2(%)                   42 non-null     float64
 4   household_consumption(%)             44 non-null     float64
 5   unemployment_rate(%)                 33 non-null     float64
 6   under_earning_percentage(%)          31 non-null     float64
 7   inflation_rate(%)                    44 non-null     float64
 8   lending_interest_rate(%)_x           38 non-null     float64
 9   fdi_net_flows(US$)                   44 non-null     float64
 10  tax_revenue_percentage(%)            26 non-null     float64
 11  state_revenues_percentage(%)         26

In [152]:
# Save the merged raw dataset so the cleaning notebook can start from a single formatted file.
merged_data.to_csv("formatted_gdp_data.csv", index=False)
